# 04. Preprocesamiento del texto

Se crea `texto_procesado` a partir de `texto_limpio`. Se sustituyen datos de contacto reconocibles por marcadores genéricos y se preservan los elementos que pueden aportar polaridad: emojis, negaciones, signos, hashtags, mayúsculas, tildes y jerga.

**Tipo utilizado:** preprocesamiento conservador basado en reglas (expresiones regulares) y sustitución no destructiva de datos de contacto.

**Aquí todavía no se tokeniza ni se crean embeddings.** Esas operaciones pertenecen al modelo y aparecen en los cuadernos 05–08.


La reproducción lee las exportaciones guardadas y escribe en `salidas/<ejecución>/TABLAS`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

**Salidas guardadas.** Las tablas, figuras y registros visibles proceden de la corrida original `corrida_03_epocas`. Esta copia incorpora las preguntas de carga de 06–08. Las salidas y los números de ejecución conservan su origen histórico. Abre el cuaderno y desplázate para exponer los resultados.

La copia pública presenta el código de esta etapa. Sus salidas por publicación se conservan en la copia de Drive.

## A. Cargar la salida limpia

- **Qué hace:** abre las cuatro columnas del cuaderno 03.
- **Para qué:** tomar `texto_limpio` como punto de partida.
- **Por qué:** no se debe volver a limpiar desde cero ni repetir transformaciones ya realizadas.


In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

from rutas import carpeta_salidas

from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter en la carpeta que contiene corrida.json.")
print("Carpeta de trabajo:", RAIZ)

import json
import re
import pandas as pd
from IPython.display import display

ENTRADA = carpeta_salidas(RAIZ) / "TABLAS/03_limpieza/datos_limpios_v4.xlsx"
datos = pd.read_excel(ENTRADA, sheet_name="datos", keep_default_na=False)
esperadas = ["id", "fuente", "texto_real", "texto_limpio"]
if list(datos.columns) != esperadas:
    raise ValueError("Ejecuta primero el cuaderno 03 actualizado.")

trabajo = datos.copy()
trabajo["texto_procesado"] = trabajo["texto_limpio"].astype(str)
display(trabajo.head(10))


## B. Sustituir enlaces

- **Qué hace:** reemplaza cada enlace por la palabra `URL`.
- **Para qué:** evitar que cadenas largas y únicas ocupen vocabulario sin aportar el contenido de la opinión.
- **Por qué:** se conserva la señal de que había un enlace, pero no su dirección específica.


In [ ]:
PATRON_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
n_url = int(trabajo["texto_procesado"].str.count(PATRON_URL).sum())
trabajo["texto_procesado"] = trabajo["texto_procesado"].str.replace(PATRON_URL, " URL ", regex=True)
print("Enlaces sustituidos:", n_url)
display(trabajo[["id", "fuente", "texto_real", "texto_procesado"]].head(10))


## C. Sustituir correos electrónicos

- **Qué hace:** reemplaza correos reconocibles por `CORREO`.
- **Para qué:** reducir exposición de datos de contacto.
- **Por qué:** la dirección concreta no es necesaria para determinar la polaridad del texto.


In [ ]:
PATRON_CORREO = re.compile(r"(?<!\w)[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}(?!\w)")
n_correo = int(trabajo["texto_procesado"].str.count(PATRON_CORREO).sum())
trabajo["texto_procesado"] = trabajo["texto_procesado"].str.replace(PATRON_CORREO, " CORREO ", regex=True)
print("Correos sustituidos:", n_correo)
display(trabajo[["id", "fuente", "texto_real", "texto_procesado"]].head(10))


## D. Sustituir usuarios y teléfonos

- **Qué hace:** cambia menciones `@usuario` por `USUARIO` y teléfonos bolivianos reconocibles por `TELEFONO`.
- **Para qué:** minimizar identificadores personales sin borrar la estructura del mensaje.
- **Por qué:** los valores concretos no aportan polaridad y pueden dificultar la generalización.


In [ ]:
PATRON_USUARIO = re.compile(r"(?<![\w@])@[A-Za-z0-9_]{2,30}\b")
PATRON_TELEFONO = re.compile(r"(?<!\w)(?:(?:\+?591)[\s.-]?)?[67]\d{7}(?!\w)")

n_usuario = int(trabajo["texto_procesado"].str.count(PATRON_USUARIO).sum())
n_telefono = int(trabajo["texto_procesado"].str.count(PATRON_TELEFONO).sum())
trabajo["texto_procesado"] = trabajo["texto_procesado"].str.replace(PATRON_USUARIO, " USUARIO ", regex=True)
trabajo["texto_procesado"] = trabajo["texto_procesado"].str.replace(PATRON_TELEFONO, " TELEFONO ", regex=True)

print("Usuarios sustituidos:", n_usuario, "| Teléfonos sustituidos:", n_telefono)
display(trabajo[["id", "fuente", "texto_real", "texto_procesado"]].head(10))


## E. Ordenar espacios, excluir vacíos y guardar

Se compactan los espacios y se excluyen las filas cuyo texto procesado queda vacío.
Se conservan los identificadores y el orden de las demás publicaciones.
El cuaderno 05 usa esta salida para verificar la procedencia de su corpus.

In [ ]:
from rutas import carpeta_salidas

trabajo["texto_procesado"] = trabajo["texto_procesado"].str.replace(r"\s+", " ", regex=True).str.strip()

vacios = trabajo["texto_procesado"].eq("")
salida = trabajo.loc[~vacios, ["id", "fuente", "texto_real", "texto_procesado"]].copy()
print(f"Filas recibidas: {len(datos)} | Vacías excluidas: {vacios.sum()} | Guardadas: {len(salida)}")

CARPETA = carpeta_salidas(RAIZ) / "TABLAS/04_preprocesamiento"
CARPETA.mkdir(parents=True, exist_ok=True)
ARCHIVO = CARPETA / "datos_preprocesados_v4.xlsx"
RESUMEN = CARPETA / "resumen_preprocesamiento_v4.json"
salida.to_excel(ARCHIVO, sheet_name="datos", index=False)

resumen = {
    "version": "v4",
    "filas_entrada": int(len(datos)),
    "vacias_excluidas": int(vacios.sum()),
    "filas": int(len(salida)),
    "url_sustituidas": n_url,
    "correos_sustituidos": n_correo,
    "usuarios_sustituidos": n_usuario,
    "telefonos_sustituidos": n_telefono,
    "conservado": ["emojis", "negaciones", "puntuación", "hashtags", "tildes", "jerga"],
    "tokenizacion": "No se realiza aquí; se aplica con el tokenizador de RoBERTuito en 05–08.",
    "columnas_salida": list(salida.columns),
}
RESUMEN.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado:", ARCHIVO)
print("Columnas finales:", list(salida.columns))
display(salida.head(10))
